# Kaggle: Streaming smart subset → Parquet

Notebook này **không tải hoặc lưu raw Amazon Reviews** vào disk Kaggle. Nó stream `.jsonl.gz` trực tiếp từ nguồn, chỉ ghi các interaction/item đã chốt vào Parquet ZSTD.

Subset là deterministic theo user (stable hash), phân tầng theo mức hoạt động, giữ trọn history của user, iterative K-core, và chronological leave-last-out. Không random theo từng review.


In [ ]:
%pip install -q 'duckdb>=1.1,<2' orjson pyarrow requests


In [ ]:
from pathlib import Path
import gzip, hashlib, io, json, shutil, gc
from collections import Counter

import duckdb, orjson, psutil, pyarrow as pa, pyarrow.parquet as pq, requests

# Giảm TARGET_* nếu Kaggle runtime/ngân sách disk chặt hơn.
TARGET_USERS = 30_000
TARGET_INTERACTIONS = 350_000
MIN_USER_DEGREE, MIN_ITEM_DEGREE = 5, 1
POSITIVE_RATING = 4.0
MIN_RATING = 1.0
ONLY_VERIFIED_PURCHASE = True
MIN_HELPFUL_VOTES = 3
SEED = '20260813'
# 1/32 user đi vào candidate pool (~700k trên raw); chỉ pool này ở RAM, không có raw trên disk.
CANDIDATE_HASH_MODULUS = 32
RESERVE_RAM_GB, MAX_DUCKDB_RAM_GB, THREADS = 4.0, 24.0, 6

WORK = Path('/kaggle/working/datn_stream_subset'); WORK.mkdir(parents=True, exist_ok=True)
REVIEW_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl.gz'
META_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Clothing_Shoes_and_Jewelry.jsonl.gz'
vm = psutil.virtual_memory(); available_gb = vm.available / 1024**3
duckdb_gb = min(MAX_DUCKDB_RAM_GB, max(0.5, (available_gb - RESERVE_RAM_GB) * 0.85))
if available_gb <= RESERVE_RAM_GB + .5: raise MemoryError('RAM khả dụng quá thấp; hãy restart Kaggle session.')
print(f'Available RAM: {available_gb:.2f} GiB | DuckDB ceiling: {duckdb_gb:.2f} GiB')


In [ ]:
def stable_hash(value: str, salt: str = SEED) -> int:
    return int.from_bytes(hashlib.blake2b(f'{salt}:{value}'.encode(), digest_size=8).digest(), 'big')

def stream_jsonl_gz(url: str, label: str):
    # Không có file raw tạm; gzip được giải nén tuần tự từ network stream.
    # Sử dụng bộ đệm 4MB để tối ưu hóa IO và giảm context switching.
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status(); response.raw.decode_content = False
        with gzip.GzipFile(fileobj=response.raw) as binary, io.BufferedReader(binary, 4 * 1024 * 1024) as buffered:
            for n, line in enumerate(buffered, 1):
                try:
                    yield orjson.loads(line)
                except orjson.JSONDecodeError:
                    continue
                if n % 5_000_000 == 0: print(f'{label}: {n:,} lines streamed')

def is_candidate(user_id: str) -> bool:
    return stable_hash(user_id, SEED + ':pool') % CANDIDATE_HASH_MODULUS == 0


## Pass 1 — đếm activity của deterministic candidate pool

Đây là điểm khác với tải raw: chỉ dictionary đếm candidate user sống trong RAM. Không có `reviews.jsonl.gz` hay bảng raw trên disk.


In [ ]:
candidate_counts = Counter()
for row in stream_jsonl_gz(REVIEW_URL, 'reviews / pass 1'):
    user_id, item_id = row.get('user_id'), row.get('parent_asin')
    rating = float(row.get('rating') or 0)
    verified = row.get('verified_purchase') is True
    helpful = int(row.get('helpful_vote') or 0)
    if (
        user_id and item_id 
        and rating >= MIN_RATING
        and (not ONLY_VERIFIED_PURCHASE or verified)
        and helpful >= MIN_HELPFUL_VOTES
        and is_candidate(user_id)
    ):
        candidate_counts[user_id] += 1

eligible = {u: n for u, n in candidate_counts.items() if n >= MIN_USER_DEGREE}
print(f'Candidate users: {len(candidate_counts):,}; eligible users: {len(eligible):,}')
# Giải phóng bộ nhớ Counter không dùng tới nữa để giải phóng RAM
del candidate_counts
gc.collect()

if len(eligible) < TARGET_USERS:
    raise RuntimeError('Candidate pool quá nhỏ. Đổi CANDIDATE_HASH_MODULUS từ 32 thành 16 rồi Run All.')


In [ ]:
def band(n): return '05_09' if n <= 9 else ('10_19' if n <= 19 else '20_plus')
groups = {name: [] for name in ('05_09', '10_19', '20_plus')}
for user_id, degree in eligible.items(): groups[band(degree)].append(user_id)
sizes = {name: len(users) for name, users in groups.items()}
total = sum(sizes.values()); floor = int(TARGET_USERS * .15)
quota = {name: min(sizes[name], max(floor, round(TARGET_USERS * sizes[name] / total))) for name in groups}
while sum(quota.values()) > TARGET_USERS:
    name = max((x for x in groups if quota[x] > min(floor, sizes[x])), key=lambda x: quota[x], default=None)
    if name is None: break
    quota[name] -= 1
while sum(quota.values()) < TARGET_USERS:
    name = max((x for x in groups if quota[x] < sizes[x]), key=lambda x: sizes[x] - quota[x], default=None)
    if name is None: break
    quota[name] += 1
selected_users = set()
for name, users in groups.items():
    selected_users.update(sorted(users, key=lambda u: stable_hash(u, SEED + ':select'))[:quota[name]])
print('Eligible bands:', sizes, '\nSelected quota:', quota, '\nSelected:', len(selected_users))


## Pass 2 — giữ lịch sử đầy đủ của selected users thành Parquet nhỏ

Review text không được mang sang dataset recommendation. Giữ text review ở giai đoạn RAG sau, khi item đã chốt; điều này tránh mang hàng GB dữ liệu không cần thiết.


In [ ]:
raw_candidate = WORK / 'candidate_interactions.parquet'
schema = pa.schema([
    ('user_id', pa.string()),
    ('item_id', pa.string()),
    ('rating', pa.float32()),
    ('timestamp', pa.int64()),
    ('verified_purchase', pa.bool_()),
    ('is_positive', pa.int32()),
    ('review_title', pa.string()),
    ('review_text', pa.string()),
    ('helpful_vote', pa.int32())
])
writer, batch, kept = pq.ParquetWriter(raw_candidate, schema, compression='zstd'), [], 0
try:
    for row in stream_jsonl_gz(REVIEW_URL, 'reviews / pass 2'):
        user_id, item_id = row.get('user_id'), row.get('parent_asin')
        if user_id not in selected_users or not item_id: continue
        rating = float(row.get('rating') or 0)
        verified = row.get('verified_purchase') is True
        helpful = int(row.get('helpful_vote') or 0)
        if rating < MIN_RATING: continue
        if ONLY_VERIFIED_PURCHASE and not verified: continue
        if helpful < MIN_HELPFUL_VOTES: continue
        batch.append({
            'user_id': user_id,
            'item_id': item_id,
            'rating': rating,
            'timestamp': int(row.get('timestamp') or 0),
            'verified_purchase': verified,
            'is_positive': 1 if rating >= POSITIVE_RATING else 0,
            'review_title': str(row.get('title') or ''),
            'review_text': str(row.get('text') or ''),
            'helpful_vote': helpful
        })
        if len(batch) >= 100_000:
            writer.write_table(pa.Table.from_pylist(batch, schema=schema)); kept += len(batch); batch.clear()
    if batch: writer.write_table(pa.Table.from_pylist(batch, schema=schema)); kept += len(batch)
finally:
    writer.close()
print(f'Wrote {kept:,} selected interaction rows ({raw_candidate.stat().st_size/1024**2:.1f} MiB)')

# Giải phóng bộ nhớ của selected_users sau khi hoàn thành pass 2
del selected_users
gc.collect()


In [ ]:
con = duckdb.connect(str(WORK / 'processing.duckdb'))
con.execute(f"SET memory_limit='{duckdb_gb:.2f}GB'"); con.execute(f'SET threads={THREADS}')
con.execute(f"SET temp_directory='{(WORK/'tmp').as_posix()}'"); con.execute('SET preserve_insertion_order=false')
con.execute(f"""CREATE OR REPLACE TABLE core AS
SELECT * FROM read_parquet('{raw_candidate.as_posix()}')
QUALIFY row_number() OVER (PARTITION BY user_id, item_id ORDER BY timestamp DESC) = 1""")

print('--- DIAGNOSTICS ---')
raw_count = con.execute('SELECT count(*) FROM core').fetchone()[0]
print(f'1. Initial core size (after loading parquet): {raw_count:,} rows')

def kcore():
    for iteration in range(20):
        before = con.execute('SELECT count(*) FROM core').fetchone()[0]
        con.execute(f"""CREATE OR REPLACE TABLE next_core AS
        WITH u AS (SELECT user_id FROM core GROUP BY 1 HAVING count(*) >= {MIN_USER_DEGREE}),
             i AS (SELECT item_id FROM core GROUP BY 1 HAVING count(*) >= {MIN_ITEM_DEGREE})
        SELECT c.* FROM core c JOIN u USING(user_id) JOIN i USING(item_id)""")
        con.execute('DROP TABLE core'); con.execute('ALTER TABLE next_core RENAME TO core')
        after = con.execute('SELECT count(*) FROM core').fetchone()[0]
        if after == before:
            print(f'   kcore stabilized at iteration {iteration}: {after:,} rows')
            return
    raise RuntimeError('K-core chưa hội tụ')

kcore()
post_kcore_count = con.execute('SELECT count(*) FROM core').fetchone()[0]
print(f'2. Core size after first kcore(): {post_kcore_count:,} rows')

con.execute(f"""CREATE OR REPLACE TABLE capped_users AS
WITH s AS (SELECT user_id, count(*) n FROM core GROUP BY 1),
r AS (SELECT s.*, row_number() OVER (PARTITION BY CASE WHEN n<10 THEN 1 WHEN n<20 THEN 2 ELSE 3 END ORDER BY hash(user_id || '{SEED}')) br FROM s),
x AS (SELECT *, sum(n) OVER (ORDER BY br, user_id) total_n FROM r) SELECT user_id FROM x WHERE total_n <= {TARGET_INTERACTIONS}""")

capped_users_count = con.execute('SELECT count(*) FROM capped_users').fetchone()[0]
print(f'3. Capped users count: {capped_users_count:,} users')

con.execute('CREATE OR REPLACE TABLE core AS SELECT c.* FROM core c JOIN capped_users USING(user_id)')
post_join_count = con.execute('SELECT count(*) FROM core').fetchone()[0]
print(f'4. Core size after joining capped_users: {post_join_count:,} rows')

kcore()
print('5. Final table print:')
print(con.execute('SELECT count(*) "rows", count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM core').fetchdf())


## Pass 3 — stream metadata, chỉ ghi metadata của item đã chốt


In [ ]:
needed_items = set(con.execute('SELECT DISTINCT item_id FROM core').fetchnumpy()['item_id'])
item_schema = pa.schema([('item_id',pa.string()),('title',pa.string()),('description',pa.string()),('features',pa.string()),('category',pa.string()),('brand',pa.string()),('price',pa.float64()),('image_url',pa.string())])
items_path = WORK / 'items_candidate.parquet'; writer = pq.ParquetWriter(items_path, item_schema, compression='zstd'); batch=[]
def text(value): return ' '.join(map(str,value)) if isinstance(value,list) else (str(value) if value is not None else None)
try:
    for row in stream_jsonl_gz(META_URL, 'metadata'):
        item_id = row.get('parent_asin')
        if item_id not in needed_items or not row.get('title'): continue
        images = row.get('images') or []; image = images[0] if isinstance(images,list) and images else {}
        batch.append({'item_id':item_id,'title':str(row['title']),'description':text(row.get('description')),'features':text(row.get('features')),
          'category':str(row.get('main_category') or ''),'brand':text(row.get('store')),'price':float(row['price']) if isinstance(row.get('price'),(int,float)) else None,
          'image_url': image.get('large') or image.get('hi_res') or image.get('thumb') if isinstance(image,dict) else None})
        if len(batch)>=25_000: writer.write_table(pa.Table.from_pylist(batch,schema=item_schema)); batch.clear()
    if batch: writer.write_table(pa.Table.from_pylist(batch,schema=item_schema))
finally: writer.close()
con.execute(f"CREATE OR REPLACE TABLE items AS SELECT * FROM read_parquet('{items_path.as_posix()}')")
con.execute('CREATE OR REPLACE TABLE core AS SELECT c.* FROM core c JOIN items USING(item_id)'); kcore()
print(con.execute('SELECT count(*) "rows", count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM core').fetchdf())

# Giải phóng bộ nhớ sau khi hoàn thành Pass 3
del needed_items
gc.collect()


In [ ]:
con.execute("""CREATE OR REPLACE TABLE split AS SELECT *,
CASE row_number() OVER (PARTITION BY user_id ORDER BY timestamp DESC, item_id) WHEN 1 THEN 'test' WHEN 2 THEN 'valid' ELSE 'train' END split FROM core""")
assert con.execute("SELECT count(*) FROM (SELECT user_id FROM split GROUP BY 1 HAVING count(DISTINCT split)<>3)").fetchone()[0] == 0
assert con.execute("SELECT count(*) FROM (SELECT user_id,max(timestamp) FILTER(WHERE split='train') a,min(timestamp) FILTER(WHERE split='valid') b,min(timestamp) FILTER(WHERE split='test') c FROM split GROUP BY 1) WHERE a>b OR b>c").fetchone()[0] == 0
for part in ('train','valid','test'):
    con.execute(f"COPY (SELECT user_id,item_id,rating,timestamp,verified_purchase,is_positive,review_title,review_text,helpful_vote FROM split WHERE split='{part}' ORDER BY user_id,timestamp) TO '{(WORK/f'{part}.parquet').as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.execute(f"COPY (SELECT i.* FROM items i JOIN (SELECT DISTINCT item_id FROM core) c USING(item_id)) TO '{(WORK/'items.parquet').as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
counts=con.execute('SELECT split,count(*) "rows",count(DISTINCT user_id) users,count(DISTINCT item_id) items FROM split GROUP BY 1').fetchdf().to_dict('records')
manifest={'source':'Amazon Reviews 2023 / Clothing_Shoes_and_Jewelry','method':'two-pass streamed stable-hash user pool; activity strata; full histories; iterative k-core; chronological leave-last-out','seed':SEED,'target_users':TARGET_USERS,'target_interactions':TARGET_INTERACTIONS,'counts':counts}
(WORK/'dataset_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf8')
con.close()
shutil.make_archive('/kaggle/working/datn_stream_subset','zip',WORK,base_dir='train.parquet')
import subprocess
subprocess.run(['zip','-j','-9','/kaggle/working/datn_stream_subset.zip',*[str(WORK/x) for x in ['train.parquet','valid.parquet','test.parquet','items.parquet','dataset_manifest.json']]],check=True)
print(counts); print('Download:', '/kaggle/working/datn_stream_subset.zip')


## Disk budget

Trong khi chạy, disk chỉ có candidate interactions vài trăm MB, metadata item vài chục MB, DuckDB temporary nhỏ và bốn Parquet cuối. Raw `.jsonl.gz` không xuất hiện trên disk. Xóa `candidate_interactions.parquet`, `items_candidate.parquet`, `processing.duckdb` sau khi tải ZIP nếu cần giải phóng thêm.
